# ChemFM · LoRA fine-tuning for molecular solubility

Fine-tune ChemFM-1B on a small ESOL subset. The complete data preparation, scaffold split, adapter training, evaluation and checkpoint export are below. Compare a newly initialized regression head with the adapted model and a training-mean baseline. This is a short execution recipe, not a reproduction of published ChemFM benchmarks.

Use a separate Python 3.12 kernel and an NVIDIA GPU. Online W&B is optional; local metrics and plots are always saved. The model weights and dataset are downloaded from pinned public revisions.


In [ ]:
import os, sys, subprocess
if os.environ.get('RECIPE_SKIP_INSTALL') != '1':
    subprocess.check_call([sys.executable,'-m','pip','install','transformers==4.46.3','peft==0.13.2','accelerate==1.2.1','rdkit','pandas','matplotlib','wandb'])
import json, hashlib, urllib.request, time, platform, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model
import matplotlib.pyplot as plt
assert torch.cuda.is_available(), 'This recipe requires CUDA. Choose a GPU kernel.'
SEED=42
np.random.seed(SEED); torch.manual_seed(SEED)
work=Path(os.environ.get('RECIPE_OUTPUT_DIR','results/chemfm'));work.mkdir(parents=True,exist_ok=True)


## Download and validate ESOL


In [ ]:
DATA_REV='455d07f3e17e880a4d980b5b0365e4a02b417e14'
url=f'https://raw.githubusercontent.com/deepchem/deepchem/{DATA_REV}/datasets/delaney-processed.csv'
raw=urllib.request.urlopen(url,timeout=60).read()
assert hashlib.sha256(raw).hexdigest()=='8c06a76f0c6487d29ab0f903e6a7a7139f189ab3c1178f159c8be8964602f189'
(work/'esol.csv').write_bytes(raw)
df=pd.read_csv(work/'esol.csv')
target='measured log solubility in mols per litre'
rows=[]
for _,row in df.iterrows():
    molecule=Chem.MolFromSmiles(str(row['smiles']).strip())
    if molecule is None or not np.isfinite(row[target]):continue
    rows.append({'smiles':Chem.MolToSmiles(molecule),'target':float(row[target]),
                 'scaffold':MurckoScaffold.MurckoScaffoldSmiles(mol=molecule)})
df=pd.DataFrame(rows).drop_duplicates('smiles').reset_index(drop=True)
# Keep each scaffold group in one partition, including the empty scaffold.
groups=[g.index.to_numpy() for _,g in df.groupby('scaffold',sort=True)]
np.random.default_rng(SEED).shuffle(groups)
split=[[],[],[]]
for group in groups:
    part=0 if len(split[0]) < .7*len(df) else (1 if len(split[1]) < .15*len(df) else 2)
    split[part].extend(group.tolist())
assert all(split), 'Empty scaffold partition; choose another split seed.'
train,val,test=[df.loc[ids].sample(frac=1,random_state=SEED).head(limit).copy() for ids,limit in zip(split,[128,32,32])]
for left,right in [(train,val),(train,test),(val,test)]:
    assert set(left.scaffold).isdisjoint(right.scaffold)
mean=float(train.target.mean()); std=float(train.target.std())
assert std>0
split_manifest={name:table.smiles.tolist() for name,table in [('train',train),('validation',val),('test',test)]}
(work/'split.json').write_text(json.dumps(split_manifest,indent=2))
print({name:len(table) for name,table in [('train',train),('validation',val),('test',test)]})


## Load ChemFM and attach trainable adapters
The original chemistry backbone is frozen. Rank-8 LoRA adapters and the regression head are trained. Standardization uses training targets only. The initial head is random, so its score is not pretrained solubility performance.


In [ ]:
MODEL_ID='ChemFM/ChemFM-1B'
MODEL_REV='f99dc2e89726539bb9cf31b2e2b4360650bac6a8'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REV)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
base=AutoModelForSequenceClassification.from_pretrained(MODEL_ID,revision=MODEL_REV,
    num_labels=1,problem_type='regression',torch_dtype=torch.bfloat16,attn_implementation='sdpa')
base.config.pad_token_id=tokenizer.pad_token_id;base.config.use_cache=False
model=get_peft_model(base,LoraConfig(task_type=TaskType.SEQ_CLS,r=8,lora_alpha=16,
    lora_dropout=.05,target_modules=['q_proj','v_proj'],modules_to_save=['score'])).cuda()
model.print_trainable_parameters()
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)

# Refuse silent SMILES truncation.
def batch(table):
    encoded=tokenizer(table.smiles.tolist(),padding=True,truncation=False,return_token_type_ids=False,return_tensors='pt')
    assert encoded['input_ids'].shape[1] <= 512, 'SMILES exceeds model context; filter explicitly.'
    return {k:v.cuda() for k,v in encoded.items()}
@torch.no_grad()
def predict(table):
    model.eval();pred=[]
    for start in range(0,len(table),4):
        pred.extend((model(**batch(table.iloc[start:start+4])).logits[:,0].float()*std+mean).cpu().tolist())
    return np.asarray(pred)
def scores(pred,table):
    error=pred-table.target.to_numpy()
    return {'rmse':float(np.sqrt(np.mean(error**2))),'mae':float(np.mean(np.abs(error)))}
before=scores(predict(test),test)
mean_baseline=scores(np.full(len(test),mean),test)


## Fine-tune and track loss
Three epochs, batch size four. Select the best checkpoint by validation RMSE; evaluate the test partition only before and after adaptation. W&B defaults to offline.


In [ ]:
import wandb
run=wandb.init(project=os.environ.get('WANDB_PROJECT','open-chem-recipes'),
    entity=os.environ.get('WANDB_ENTITY'),name='chemfm-esol-lora',
    mode=os.environ.get('WANDB_MODE','online' if os.environ.get('WANDB_API_KEY') else 'offline'),
    config={'model':MODEL_ID,'revision':MODEL_REV,'epochs':3,'trainable_parameters':trainable,'seed':SEED})
optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=2e-4)
curve=[];best=float('inf');best_state=None;step=0;started=time.monotonic()
for epoch in range(3):
    shuffled=train.sample(frac=1,random_state=SEED+epoch)
    model.train()
    for start in range(0,len(shuffled),4):
        part=shuffled.iloc[start:start+4]
        labels=torch.tensor((part.target.to_numpy()-mean)/std,dtype=torch.float32,device='cuda')
        prediction=model(**batch(part)).logits[:,0].float()
        loss=(prediction-labels).square().mean()
        assert torch.isfinite(loss),'Nonfinite training loss'
        optimizer.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1);optimizer.step()
        curve.append({'step':step,'epoch':epoch+1,'loss':float(loss.item())})
        run.log({'train/loss':float(loss.item())},step=step);step+=1
    validation=scores(predict(val),val)
    run.log({'validation/rmse':validation['rmse']},step=step);step+=1
    if validation['rmse']<best:
        best=validation['rmse']
        best_state={k:p.detach().cpu().clone() for k,p in model.named_parameters() if p.requires_grad}
assert best_state is not None
with torch.no_grad():
    for name,p in model.named_parameters():
        if name in best_state:p.copy_(best_state[name].to(p.device))
after_predictions=predict(test);after=scores(after_predictions,test)
model.save_pretrained(work/'adapter');tokenizer.save_pretrained(work/'adapter')
(work/'adapter'/'target_scaler.json').write_text(json.dumps({'mean':mean,'std':std}))
metrics={'before_random_head':before,'after':after,'training_mean_baseline':mean_baseline,
         'validation_best_rmse':best,'trainable_parameters':trainable,'curve':curve,
         'wall_seconds':time.monotonic()-started,'gpu':torch.cuda.get_device_name(0),
         'split_sizes':{k:len(v) for k,v in split_manifest.items()}}
result={'metrics':metrics,'model':MODEL_ID,'model_revision':MODEL_REV,'dataset_revision':DATA_REV,
        'python':platform.python_version(),'packages':{p:importlib.metadata.version(p) for p in ['torch','transformers','peft','rdkit']}}
(work/'result.json').write_text(json.dumps(result,indent=2))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot([p['step'] for p in curve],[p['loss'] for p in curve]);axes[0].set(xlabel='Step',ylabel='Standardized MSE',title='ChemFM LoRA training')
axes[1].scatter(test.target,after_predictions);limits=[min(test.target.min(),after_predictions.min()),max(test.target.max(),after_predictions.max())]
axes[1].plot(limits,limits,'--',color='gray');axes[1].set(xlabel='Measured logS',ylabel='Predicted logS',title='Held-out scaffold partition')
fig.tight_layout();fig.savefig(work/'result.png',dpi=140);run.log({'results':wandb.Image(fig)});run.summary.update(after);run.finish();plt.show()
print(json.dumps({k:metrics[k] for k in ['before_random_head','after','training_mean_baseline']},indent=2))


## Interpretation and reuse

Use `adapter/` together with the exact base revision and `target_scaler.json`. Different GPU kernels can change results. The small scaffold split is harder than a random split but is not a generalization guarantee; pretraining overlap is unknown. Run multiple seeds and larger datasets before comparing models. The original ChemFM source repository and model weights declare different licenses; see the repository notices.
